# Milestone 4: Interactive Graph Dashboard
**AI Knowledge Graph Builder for Enterprise Intelligence**

| Component | Tool |
|---|---|
| Dashboard | Streamlit |
| Graph Visualization | PyVis + Plotly |
| Semantic Search | LangChain + FAISS + Groq |
| Graph DB | Neo4j (Milestone 2) |
| Deployment | ngrok |


# PART 1: Install & Setup

## 1.1 Install Dependencies

In [1]:
!pip install streamlit pyvis plotly pyngrok neo4j langchain langchain-groq langchain-community langchain-core sentence-transformers faiss-cpu pandas -q

print('All dependencies installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source o

## 1.2 Configuration

In [2]:
# ================================================================
# CONFIGURATION - UPDATE IF NEEDED
# ================================================================

NEO4J_URI      = "your_api_key"
NEO4J_USER     = "your_username"
NEO4J_PASSWORD = "your_password"

GROQ_API_KEY   = "your_api_key"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL       = "llama-3.3-70b-versatile"
TOP_K_RESULTS   = 10

NGROK_TOKEN = "your_api_key"  # Get FREE at https://ngrok.com

print("Configuration set!")


Configuration set!


# PART 2: Write Dashboard Files

## 2.1 Write styles.css

In [3]:
%%writefile styles.css
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');
* { font-family: 'Inter', sans-serif; }
.stApp {
    background: linear-gradient(135deg, #0a0a0f 0%, #0d1117 50%, #0a0f1e 100%);
    color: #e2e8f0;
}
[data-testid="stSidebar"] {
    background: rgba(13, 17, 23, 0.95) !important;
    border-right: 1px solid rgba(99, 102, 241, 0.3);
}
.glass-card {
    background: rgba(255, 255, 255, 0.03);
    border: 1px solid rgba(99, 102, 241, 0.2);
    border-radius: 16px;
    padding: 24px;
    backdrop-filter: blur(10px);
    margin-bottom: 16px;
    transition: all 0.3s ease;
}
.glass-card:hover {
    border-color: rgba(99, 102, 241, 0.5);
    background: rgba(255, 255, 255, 0.05);
    transform: translateY(-2px);
    box-shadow: 0 8px 32px rgba(99, 102, 241, 0.15);
}
.metric-card {
    background: linear-gradient(135deg, rgba(99, 102, 241, 0.1), rgba(139, 92, 246, 0.1));
    border: 1px solid rgba(99, 102, 241, 0.3);
    border-radius: 12px;
    padding: 20px;
    text-align: center;
}
.gradient-title {
    background: linear-gradient(135deg, #6366f1, #8b5cf6, #06b6d4);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.5rem;
    font-weight: 700;
    text-align: center;
}
.stTabs [data-baseweb="tab-list"] {
    background: rgba(255,255,255,0.03);
    border-radius: 12px;
    padding: 4px;
    gap: 4px;
}
.stTabs [data-baseweb="tab"] { border-radius: 8px; color: #94a3b8; font-weight: 500; }
.stTabs [aria-selected="true"] { background: linear-gradient(135deg, #6366f1, #8b5cf6) !important; color: white !important; }
.stButton > button {
    background: linear-gradient(135deg, #6366f1, #8b5cf6);
    color: white; border: none; border-radius: 10px;
    padding: 10px 24px; font-weight: 600; transition: all 0.3s; width: 100%;
}
.stButton > button:hover { transform: translateY(-2px); box-shadow: 0 8px 25px rgba(99, 102, 241, 0.4); }
.stTextInput > div > div > input {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(99, 102, 241, 0.3) !important;
    border-radius: 10px !important; color: #e2e8f0 !important;
}
.result-card {
    background: rgba(99, 102, 241, 0.08);
    border: 1px solid rgba(99, 102, 241, 0.25);
    border-radius: 12px; padding: 16px; margin: 8px 0;
}
.tag { display: inline-block; padding: 4px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; margin: 2px; }
.tag-remote   { background: rgba(16,185,129,0.15); border: 1px solid #10b981; color: #10b981; }
.tag-onsite   { background: rgba(239,68,68,0.15);  border: 1px solid #ef4444; color: #ef4444; }
.tag-hybrid   { background: rgba(245,158,11,0.15); border: 1px solid #f59e0b; color: #f59e0b; }
.tag-premium  { background: rgba(99,102,241,0.15); border: 1px solid #6366f1; color: #6366f1; }
.tag-skill    { background: rgba(6,182,212,0.15);  border: 1px solid #06b6d4; color: #06b6d4; }
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: rgba(255,255,255,0.02); }
::-webkit-scrollbar-thumb { background: rgba(99,102,241,0.4); border-radius: 3px; }
print("styles.css written!")


Writing styles.css


## 2.2 Write graph_utils.py

In [4]:
%%writefile graph_utils.py
from neo4j import GraphDatabase
from dataclasses import dataclass
from typing import List
import pandas as pd

@dataclass
class Job:
    job_id: str
    category: str
    workplace: str
    employment_type: str
    priority_class: str
    demand_score: float
    city: str
    country: str
    region: str
    department: str
    department_category: str
    is_active: bool
    text_description: str

def load_jobs_from_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    query = """
    MATCH (j:Job)-[:LOCATED_IN]->(l:Location),
          (j)-[:IN_DEPARTMENT]->(d:Department),
          (j)-[:BELONGS_TO]->(c:Category)
    RETURN
        j.id AS job_id,
        c.name AS category,
        j.workplace AS workplace,
        j.employment_type AS employment_type,
        j.priority_class AS priority_class,
        j.demand_score AS demand_score,
        l.city AS city,
        l.country AS country,
        l.region AS region,
        d.name AS department,
        d.category AS department_category,
        j.is_active AS is_active
    """
    jobs = []
    with driver.session() as session:
        for record in session.run(query):
            text = (
                f"Job: {record['category']}\n"
                f"Location: {record['city']}, {record['country']} ({record['region']} region)\n"
                f"Work: {record['workplace']} {record['employment_type']}\n"
                f"Department: {record['department']} ({record['department_category']})\n"
                f"Priority: {record['priority_class']}\n"
                f"Demand Score: {record['demand_score']:.1f}/100"
            )
            jobs.append(Job(
                job_id=record['job_id'],
                category=record['category'],
                workplace=record['workplace'],
                employment_type=record['employment_type'],
                priority_class=record['priority_class'],
                demand_score=float(record['demand_score']),
                city=record['city'],
                country=record['country'],
                region=record['region'],
                department=record['department'],
                department_category=record['department_category'],
                is_active=bool(record['is_active']),
                text_description=text
            ))
    driver.close()
    return jobs

def load_graph_data(uri, user, password):
    """Load nodes and edges using elementId for consistent matching"""
    driver = GraphDatabase.driver(uri, auth=(user, password))
    nodes = []
    edges = []

    try:
        # Load nodes — use elementId() as unique key
        with driver.session() as session:
            result = session.run("""
                MATCH (n)
                RETURN
                    elementId(n)    AS eid,
                    labels(n)[0]    AS label,
                    coalesce(n.id, n.name, elementId(n)) AS display_id
                LIMIT 1300
            """)
            for r in result:
                eid = str(r["eid"])
                nodes.append({
                    "id"    : eid,
                    "label" : r["label"],
                    "name"  : str(r["display_id"])
                })

        # Load edges — use elementId() for src and tgt
        with driver.session() as session:
            result = session.run("""
                MATCH (a)-[r]->(b)
                RETURN
                    elementId(a) AS src,
                    elementId(b) AS tgt,
                    type(r)      AS rel
                LIMIT 5243
            """)
            for r in result:
                edges.append({
                    "src": str(r["src"]),
                    "tgt": str(r["tgt"]),
                    "rel": r["rel"]
                })
    finally:
        driver.close()

    return nodes, edges

def load_stats(uri, user, password):
    """Load graph statistics"""
    driver = GraphDatabase.driver(uri, auth=(user, password))
    stats = {}
    try:
        with driver.session() as session:
            result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt")
            stats["nodes"] = {r["label"]: r["cnt"] for r in result}
        with driver.session() as session:
            result = session.run("MATCH ()-[r]->() RETURN type(r) AS rel, count(r) AS cnt")
            stats["edges"] = {r["rel"]: r["cnt"] for r in result}
        with driver.session() as session:
            result = session.run("""
                MATCH (j:Job)-[:REQUIRES]->(s:Skill)
                RETURN s.name AS skill, count(j) AS cnt
                ORDER BY cnt DESC LIMIT 20
            """)
            stats["top_skills"] = [(r["skill"], r["cnt"]) for r in result]
    finally:
        driver.close()
    return stats

print("graph_utils.py written!")

Writing graph_utils.py


## 2.3 Write search_utils.py

In [5]:
%%writefile search_utils.py
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

def build_rag_pipeline(jobs, groq_api_key, embedding_model, llm_model, top_k):
    documents = []
    for job in jobs:
        doc = Document(
            page_content=job.text_description,
            metadata={
                "job_id": job.job_id, "category": job.category,
                "workplace": job.workplace, "employment_type": job.employment_type,
                "priority_class": job.priority_class, "demand_score": job.demand_score,
                "city": job.city, "country": job.country, "region": job.region,
                "department": job.department, "department_category": job.department_category,
            }
        )
        documents.append(doc)

    embeddings_model = HuggingFaceEmbeddings(
        model_name=embedding_model,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    vectorstore = FAISS.from_documents(documents=documents, embedding=embeddings_model)
    retriever   = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": top_k, "fetch_k": 30}
    )

    llm = ChatGroq(api_key=groq_api_key, model_name=llm_model, temperature=0.3, max_tokens=512)

    RAG_PROMPT = PromptTemplate(
        input_variables=["context", "question"],
        template="""You are an intelligent job search assistant for an enterprise knowledge graph.
Use these retrieved job listings to answer:
{context}
Question: {question}
Give a helpful 2-3 sentence answer mentioning how many jobs found, key locations and patterns:"""
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | RAG_PROMPT | llm | StrOutputParser()
    )

    return rag_chain, retriever

def run_search(rag_chain, retriever, query):
    answer    = rag_chain.invoke(query)
    retrieved = retriever.invoke(query)
    results   = [doc.metadata for doc in retrieved]
    return answer, results

print("search_utils.py written!")


Writing search_utils.py


## 2.4 Write app.py — Main Streamlit Dashboard

In [6]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pyvis.network import Network
import os, json, tempfile

st.set_page_config(
    page_title="AI Knowledge Graph Dashboard",
    page_icon="brain",
    layout="wide",
    initial_sidebar_state="expanded"
)

with open("styles.css") as f:
    st.markdown(f"<style>{f.read()}</style>", unsafe_allow_html=True)

NEO4J_URI      = "your_api_key"
NEO4J_USER     = "your_username"
NEO4J_PASSWORD = "your_password"
GROQ_API_KEY   = "your_api_key"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL       = "llama-3.3-70b-versatile"
TOP_K_RESULTS   = 10

@st.cache_resource(show_spinner="Connecting to Neo4j...")
def load_data():
    from graph_utils import load_jobs_from_neo4j, load_graph_data, load_stats
    jobs         = load_jobs_from_neo4j(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    nodes, edges = load_graph_data(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    stats        = load_stats(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    return jobs, nodes, edges, stats

@st.cache_resource(show_spinner="Building RAG pipeline...")
def load_rag(_job_ids):
    jobs, _, _, _ = load_data()
    from search_utils import build_rag_pipeline
    return build_rag_pipeline(jobs, GROQ_API_KEY, EMBEDDING_MODEL, LLM_MODEL, TOP_K_RESULTS)

jobs, nodes, edges, stats = load_data()
rag_chain, retriever = load_rag(tuple(j.job_id for j in jobs))

df = pd.DataFrame([{
    "Job ID": j.job_id, "Category": j.category, "Workplace": j.workplace,
    "Employment": j.employment_type, "Priority": j.priority_class,
    "Demand Score": j.demand_score, "City": j.city, "Country": j.country,
    "Region": j.region, "Department": j.department_category,
} for j in jobs])

# ── Sidebar ──
with st.sidebar:
    st.markdown("<div style='text-align:center;padding:20px 0;'><div style='font-size:2rem;'>Knowledge Graph</div><div style='color:#64748b;font-size:0.8rem;'>Enterprise Intelligence</div></div>", unsafe_allow_html=True)
    st.divider()
    st.markdown("### Filters")
    cat_filter = st.multiselect("Category",  sorted(df["Category"].unique()),  default=[])
    wp_filter  = st.multiselect("Workplace", sorted(df["Workplace"].unique()),  default=[])
    reg_filter = st.multiselect("Region",    sorted(df["Region"].unique()),     default=[])
    pri_filter = st.multiselect("Priority",  sorted(df["Priority"].unique()),   default=[])
    st.divider()
    total_nodes = sum(stats["nodes"].values())
    total_edges = sum(stats["edges"].values())
    st.markdown(f"<div style='text-align:center;'><div style='color:#6366f1;font-size:1.8rem;font-weight:700;'>{total_nodes}</div><div style='color:#94a3b8;font-size:0.8rem;'>Total Nodes</div><div style='color:#8b5cf6;font-size:1.8rem;font-weight:700;margin-top:8px;'>{total_edges}</div><div style='color:#94a3b8;font-size:0.8rem;'>Total Relationships</div><div style='color:#06b6d4;font-size:1.8rem;font-weight:700;margin-top:8px;'>{len(jobs)}</div><div style='color:#94a3b8;font-size:0.8rem;'>Jobs Indexed</div></div>", unsafe_allow_html=True)

fdf = df.copy()
if cat_filter: fdf = fdf[fdf["Category"].isin(cat_filter)]
if wp_filter:  fdf = fdf[fdf["Workplace"].isin(wp_filter)]
if reg_filter: fdf = fdf[fdf["Region"].isin(reg_filter)]
if pri_filter: fdf = fdf[fdf["Priority"].isin(pri_filter)]

# ── Header ──
st.markdown("<div class='gradient-title'>AI Knowledge Graph Dashboard</div><div style='text-align:center;color:#64748b;margin-bottom:24px;'>Milestone 4 | Interactive Graph Exploration | RAG Semantic Search</div>", unsafe_allow_html=True)

# ── Top Metrics ──
c1,c2,c3,c4,c5 = st.columns(5)
for col,label,value,color in [
    (c1,"Jobs",        stats["nodes"].get("Job",0),       "#6366f1"),
    (c2,"Locations",   stats["nodes"].get("Location",0),  "#8b5cf6"),
    (c3,"Departments", stats["nodes"].get("Department",0),"#06b6d4"),
    (c4,"Skills",      stats["nodes"].get("Skill",0),     "#10b981"),
    (c5,"Relationships",total_edges,                       "#f59e0b"),
]:
    col.markdown(f"<div class='metric-card'><div style='color:{color};font-size:2rem;font-weight:700;'>{value}</div><div style='color:#94a3b8;font-size:0.85rem;'>{label}</div></div>", unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

tab1,tab2,tab3,tab4,tab5 = st.tabs([
    "🌐　Graph Explorer　",
    "📊　Analytics　",
    "🔍　Semantic Search　",
    "💼　Job Explorer　",
    "🗺️　Insights　"
])

# ── TAB 1: Graph Explorer ──
with tab1:
    st.markdown("### Interactive Knowledge Graph")
    st.caption("Hover over nodes | Drag to explore | Scroll to zoom")
    col_ctrl, col_graph = st.columns([1,4])
    with col_ctrl:
        st.markdown("**Display Options**")
        show_jobs   = st.checkbox("Jobs",        True)
        show_locs   = st.checkbox("Locations",   True)
        show_depts  = st.checkbox("Departments", True)
        show_cats   = st.checkbox("Categories",  True)
        show_skills = st.checkbox("Skills",      True)
        node_limit = st.slider("Max nodes", 50, 1300, 500)
        physics     = st.selectbox("Physics", ["forceAtlas2Based","barnesHut","repulsion"])
        st.markdown("**Node Colors**")
        st.markdown("Blue=Job | Green=Location | Orange=Department | Red=Category | Cyan=Skill")
    with col_graph:
        label_filter = []
        if show_jobs:   label_filter.append("Job")
        if show_locs:   label_filter.append("Location")
        if show_depts:  label_filter.append("Department")
        if show_cats:   label_filter.append("Category")
        if show_skills: label_filter.append("Skill")

        color_map = {
            "Job":        "#6366f1",
            "Location":   "#10b981",
            "Department": "#f59e0b",
            "Category":   "#ef4444",
            "Skill":      "#06b6d4"
        }

        net = Network(height="600px", width="100%", bgcolor="#0d1117", font_color="#e2e8f0")

        options = """
{
  "physics": {
    "enabled": true,
    "forceAtlas2Based": {
      "gravitationalConstant": -50,
      "springLength": 100,
      "springConstant": 0.08
    },
    "solver": "forceAtlas2Based"
  },
  "nodes": {
    "borderWidth": 2,
    "shadow": true,
    "font": {"color": "#e2e8f0", "size": 12}
  },
  "edges": {
    "color": {"color": "#6366f1", "opacity": 0.8},
    "width": 2,
    "smooth": {"type": "continuous"},
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.5}}
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "navigationButtons": true
  }
}
"""
        net.set_options(options)

        added_nodes = set()
        filtered_nodes = [n for n in nodes if n["label"] in label_filter][:node_limit]
        # Add ALL nodes first before adding edges
        for node in filtered_nodes:
            nid = str(node["id"]) if node["id"] else None
            if nid and nid not in added_nodes:
                color = color_map.get(node["label"], "#6366f1")
                size  = 30 if node["label"] == "Job" else 20 if node["label"] == "Skill" else 25
                net.add_node(nid, label=str(nid)[:15], color=color, size=size, title=f"{node['label']}: {nid}")
                added_nodes.add(nid)


        edge_count = 0
        for edge in edges:
            src = str(edge["src"]) if edge["src"] else None
            tgt = str(edge["tgt"]) if edge["tgt"] else None
            if src and tgt and src in added_nodes and tgt in added_nodes:
                rel_color = "#6366f1" if edge["rel"] == "REQUIRES" else "#94a3b8"
                net.add_edge(
                    src, tgt,
                    title=edge["rel"],
                    color=rel_color,
                    width=2,
                    arrows="to"
                )
                edge_count += 1

        st.caption(f"Showing {len(added_nodes)} nodes and {edge_count} relationships")

        with tempfile.NamedTemporaryFile(delete=False, suffix=".html") as f:
            net.save_graph(f.name)
            html = open(f.name).read()

        st.components.v1.html(html, height=620, scrolling=False)

# ── TAB 2: Analytics ──
with tab2:
    st.markdown("### Graph Analytics")
    r1c1,r1c2 = st.columns(2)
    with r1c1:
        nd = pd.DataFrame(list(stats["nodes"].items()), columns=["Type","Count"])
        fig = px.pie(nd, names="Type", values="Count", title="Node Distribution",
                     color_discrete_sequence=["#6366f1","#10b981","#f59e0b","#ef4444","#06b6d4"], hole=0.5)
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    with r1c2:
        ed = pd.DataFrame(list(stats["edges"].items()), columns=["Type","Count"])
        fig = px.bar(ed, x="Type", y="Count", title="Relationship Types",
                     color="Count", color_continuous_scale=["#6366f1","#8b5cf6","#06b6d4"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    r2c1,r2c2 = st.columns(2)
    with r2c1:
        cc = fdf["Category"].value_counts().reset_index(); cc.columns=["Category","Count"]
        fig = px.bar(cc, x="Category", y="Count", title=f"Jobs by Category ({len(fdf)} total)",
                     color="Count", color_continuous_scale=["#6366f1","#8b5cf6","#a78bfa"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    with r2c2:
        wc = fdf["Workplace"].value_counts().reset_index(); wc.columns=["Workplace","Count"]
        fig = px.pie(wc, names="Workplace", values="Count", title="Work Arrangement",
                     color_discrete_sequence=["#10b981","#ef4444","#f59e0b"], hole=0.4)
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    if stats["top_skills"]:
        st.markdown("### Top 20 Most Required Skills")
        sd = pd.DataFrame(stats["top_skills"], columns=["Skill","Jobs"])
        fig = px.bar(sd, x="Jobs", y="Skill", orientation="h",
                     title="Skills extracted via LLM-based NER (Milestone 2)",
                     color="Jobs", color_continuous_scale=["#6366f1","#8b5cf6","#06b6d4"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                          font_color="#e2e8f0", height=600, yaxis={"categoryorder":"total ascending"})
        st.plotly_chart(fig, use_container_width=True)

# ── TAB 3: Semantic Search ──
with tab3:
    st.markdown("### RAG-Powered Semantic Search")
    st.caption("Powered by LangChain + FAISS + Groq Llama 3")
    st.markdown("**Try these queries:**")
    suggestions = [
        "Remote Data Scientist jobs in India",
        "Premium Software Developer in Europe",
        "Full time HR jobs in Asia Pacific",
        "Cloud computing high demand jobs",
        "Hybrid Business Analyst in US",
    ]
    scols = st.columns(len(suggestions))
    for i,(col,sug) in enumerate(zip(scols,suggestions)):
        if col.button(sug, key=f"sug_{i}"):
            st.session_state["search_query"] = sug
    query = st.text_input(
        "Ask anything about the job market:",
        value=st.session_state.get("search_query",""),
        placeholder="e.g. Show me remote full-time jobs in India..."
    )
    if st.button("Search", key="search_btn") and query:
        with st.spinner("Groq Llama 3 is thinking..."):
            from search_utils import run_search
            answer, results = run_search(rag_chain, retriever, query)
        st.markdown(f"<div class='glass-card'><div style='color:#6366f1;font-weight:700;margin-bottom:8px;'>AI Answer</div><div style='color:#e2e8f0;line-height:1.6;'>{answer}</div></div>", unsafe_allow_html=True)
        st.markdown(f"### Top {len(results)} Matching Jobs")
        for i,job in enumerate(results,1):
            wp_color  = {"Remote":"tag-remote","On-Site":"tag-onsite","Hybrid":"tag-hybrid"}.get(job.get("workplace",""),"tag-skill")
            pri_color = "tag-premium" if job.get("priority_class","")=="Premium" else "tag-skill"
            st.markdown(f"<div class='result-card'><div style='font-weight:700;color:#e2e8f0;margin-bottom:8px;'>{i}. {job.get('category','N/A')} - {job.get('city','N/A')}, {job.get('country','N/A')}</div><span class='tag {wp_color}'>{job.get('workplace','N/A')}</span><span class='tag {pri_color}'>{job.get('priority_class','N/A')}</span><span class='tag tag-skill'>{job.get('employment_type','N/A')}</span><span class='tag tag-hybrid'>{job.get('department_category','N/A')}</span><div style='color:#64748b;font-size:0.8rem;margin-top:8px;'>Demand: {job.get('demand_score',0):.1f}/100 | Region: {job.get('region','N/A')}</div></div>", unsafe_allow_html=True)

# ── TAB 4: Job Explorer ──
with tab4:
    st.markdown(f"### Job Explorer - {len(fdf)} jobs found")
    st.caption("Use sidebar filters to drill down")
    q1,q2,q3,q4 = st.columns(4)
    for col,label,value,color in [
        (q1,"Remote Jobs",  len(fdf[fdf["Workplace"]=="Remote"]), "#10b981"),
        (q2,"Premium Jobs", len(fdf[fdf["Priority"]=="Premium"]), "#6366f1"),
        (q3,"Countries",    fdf["Country"].nunique(),              "#06b6d4"),
        (q4,"Avg Demand",   f"{fdf['Demand Score'].mean():.1f}",  "#f59e0b"),
    ]:
        col.markdown(f"<div class='metric-card'><div style='color:{color};font-size:1.6rem;font-weight:700;'>{value}</div><div style='color:#94a3b8;font-size:0.8rem;'>{label}</div></div>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)
    fc1,fc2 = st.columns(2)
    with fc1:
        rd = fdf["Region"].value_counts().reset_index(); rd.columns=["Region","Count"]
        fig = px.bar(rd, x="Region", y="Count", title="Jobs by Region",
                     color="Count", color_continuous_scale=["#6366f1","#06b6d4"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    with fc2:
        fig = px.histogram(fdf, x="Demand Score", nbins=20, title="Demand Score Distribution",
                           color_discrete_sequence=["#8b5cf6"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    st.markdown("### Job Listings")
    st.dataframe(fdf.style.background_gradient(subset=["Demand Score"], cmap="Blues"),
                 use_container_width=True, height=400)

# ── TAB 5: Insights ──
with tab5:
    st.markdown("### Global Insights")
    country_counts = fdf["Country"].value_counts().reset_index(); country_counts.columns=["Country","Jobs"]
    fig = px.choropleth(country_counts, locations="Country", locationmode="country names",
                        color="Jobs", title="Job Distribution by Country",
                        color_continuous_scale=["#1e1b4b","#6366f1","#a78bfa","#c4b5fd"])
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0",
                      geo=dict(bgcolor="rgba(0,0,0,0)", showframe=False,
                               showcoastlines=True, coastlinecolor="#334155",
                               showland=True, landcolor="#1e293b",
                               showocean=True, oceancolor="#0f172a"),
                      height=650,
                      margin=dict(l=0, r=0, t=40, b=0))
    st.plotly_chart(fig, use_container_width=True)
    ic1,ic2 = st.columns(2)
    with ic1:
        dd = fdf["Department"].value_counts().reset_index(); dd.columns=["Department","Count"]
        fig = px.treemap(dd, path=["Department"], values="Count", title="Jobs by Department",
                         color="Count", color_continuous_scale=["#1e1b4b","#6366f1","#06b6d4"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    with ic2:
        ec = fdf.groupby(["Employment","Category"]).size().reset_index(name="Count")
        fig = px.sunburst(ec, path=["Employment","Category"], values="Count",
                          title="Employment Type to Category",
                          color_discrete_sequence=["#6366f1","#8b5cf6","#06b6d4","#10b981","#f59e0b"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
        st.plotly_chart(fig, use_container_width=True)
    hm = fdf.groupby(["Category","Priority"]).size().unstack(fill_value=0)
    fig = px.imshow(hm, title="Priority Heatmap by Category",
                    color_continuous_scale=["#0f172a","#6366f1","#a78bfa"], aspect="auto")
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0")
    st.plotly_chart(fig, use_container_width=True)

st.markdown("<div style='text-align:center;padding:20px;color:#475569;border-top:1px solid rgba(99,102,241,0.2);margin-top:40px;'>AI Knowledge Graph Builder | Milestone 4 | LangChain + FAISS + Groq + Neo4j + Streamlit</div>", unsafe_allow_html=True)


Writing app.py


# PART 3: Deploy Dashboard

## 3.1 Setup ngrok Token

In [7]:
# Sign up -> Dashboard -> Authtoken -> Copy

NGROK_TOKEN = "your_api_key"

print("Token set! Run next cell to launch.")


Token set! Run next cell to launch.


## 3.2 Launch Dashboard

In [8]:
import subprocess, threading, time
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
ngrok.kill()
time.sleep(2)

def run_streamlit():
    subprocess.run([
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--theme.base", "dark",
        "--theme.primaryColor", "#6366f1",
        "--theme.backgroundColor", "#0a0a0f",
        "--theme.secondaryBackgroundColor", "#0d1117",
        "--theme.textColor", "#e2e8f0",
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)

tunnel = ngrok.connect(8501)
url    = tunnel.public_url

print("=" * 60)
print("  DASHBOARD IS LIVE!")
print("=" * 60)
print(f"\n  Public URL: {url}")
print(f"\n  Share this URL with your mentor and panel!")
print("=" * 60)


  DASHBOARD IS LIVE!

  Public URL: https://aubrey-overbitter-prehistorically.ngrok-free.dev

  Share this URL with your mentor and panel!


## 3.3 Stop Dashboard (when done)

In [ ]:
from pyngrok import ngrok
import subprocess
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
print("Dashboard stopped!")


Dashboard stopped!


---
# MILESTONE 4 COMPLETE!

## What You Built

| Tab | Feature |
|---|---|
| Graph Explorer | PyVis animated knowledge graph - all 1298 nodes |
| Analytics | Plotly charts - node distribution, relationships, skills |
| Semantic Search | LangChain + FAISS + Groq RAG pipeline |
| Job Explorer | Filterable job table with drill-down |
| Insights | World map, treemap, sunburst, heatmap |

## Tech Stack
| Component | Tool |
|---|---|
| Dashboard | Streamlit |
| Graph Visualization | PyVis + Plotly |
| RAG Search | LangChain + FAISS + Groq |
| Graph DB | Neo4j (Milestone 2) |
| Deployment | ngrok |
| **Total Cost** | **$0.00** |
